In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from collections import Counter
import math

In [ ]:
# Balanced event trace dataset
df = pd.read_csv(
    "/content/drive/MyDrive/Log_Anamoly_Detection/dataset/balanced_log_dataset.csv"
)

# Transformer output (only anomalous rows)
anomaly_results = pd.read_csv(
    "/content/drive/MyDrive/Log_Anamoly_Detection/results/detected_anomalies.csv"
)

# HDFS log templates (EventId -> EventTemplate)
templates_df = pd.read_csv(
    "/content/drive/MyDrive/Log_Anamoly_Detection/dataset/HDFS.log_templates.csv"
)

print("Templates columns:", templates_df.columns.tolist())
print("Logs shape:", df.shape)
print("Anomaly results shape:", anomaly_results.shape)
print(anomaly_results.columns.tolist())

Templates columns: ['EventId', 'EventTemplate']
Logs shape: (33676, 6)
Anomaly results shape: (3378, 5)
['BlockId', 'Predicted_Label', 'Features', 'TimeInterval', 'Latency']


In [ ]:
print("\nLabel values:", df["Label"].unique())


Label values: ['Fail' 'Success']


In [ ]:
anomalous_blocks = anomaly_results["BlockId"].unique().tolist()
print("Number of anomalous blocks:", len(anomalous_blocks))

Number of anomalous blocks: 3378


In [ ]:
anomalous_logs = df[df["BlockId"].isin(anomalous_blocks)].copy()
print(f"Anomalous log rows     : {len(anomalous_logs)}")

Anomalous log rows     : 3378


In [ ]:
print(df.head())
print(df.columns)

                   BlockId    Label  Type  \
0  blk_8462687553742484299     Fail   5.0   
1  blk_9151689281583792216     Fail   1.0   
2  blk_5446956458314793726  Success   NaN   
3  blk_6632164064180342420  Success   NaN   
4  blk_-427522761434588674     Fail   4.0   

                                            Features  \
0  [E5,E5,E5,E22,E11,E9,E11,E9,E11,E9,E26,E26,E26...   
1  [E22,E5,E5,E5,E26,E26,E11,E9,E11,E9,E11,E9,E27...   
2    [E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26]   
3    [E5,E5,E22,E5,E11,E9,E11,E9,E26,E26,E26,E11,E9]   
4  [E5,E5,E22,E5,E9,E11,E9,E11,E9,E26,E26,E11,E26...   

                                        TimeInterval  Latency  
0  [0.0, 0.0, 0.0, 70.0, 0.0, 0.0, 0.0, 0.0, 0.0,...     7156  
1  [0.0, 0.0, 0.0, 18.0, 0.0, 0.0, 0.0, 0.0, 0.0,...    39300  
2  [0.0, 0.0, 1.0, 47.0, 0.0, 0.0, 0.0, 0.0, 0.0,...       48  
3  [0.0, 0.0, 1.0, 46.0, 0.0, 0.0, 0.0, 0.0, 0.0,...       48  
4  [0.0, 0.0, 3.0, 35.0, 0.0, 0.0, 0.0, 0.0, 0.0,...    48073  
Index(['

In [ ]:
def parse_event_sequence(seq):
    seq = seq.strip("[]")
    return [e.strip() for e in seq.split(",")]

In [ ]:
anomalous_logs["EventSequence"] = anomalous_logs["Features"].apply(parse_event_sequence)
anomalous_logs["EventCounts"] = anomalous_logs["EventSequence"].apply(lambda x: dict(Counter(x)))

In [ ]:
max_repeats       = anomalous_logs["EventCounts"].apply(lambda d: max(d.values()))
REPEAT_THRESHOLD  = math.ceil(max_repeats.quantile(0.75))   

LATENCY_THRESHOLD = int(df["Latency"].quantile(0.95))       

print(f"Repeat threshold  : > {REPEAT_THRESHOLD} occurrences of any single event")
print(f"Latency threshold : > {LATENCY_THRESHOLD} ms")

Repeat threshold  : > 4 occurrences of any single event
Latency threshold : > 50937 ms


In [ ]:
# ── Anomaly type definitions ──────────────────────────────────
#
# Each type has 4 things the LLM needs:
#   - summary   : what is happening in this block
#   - root_cause: why it is happening
#   - details   : dynamic detail filled from the actual row data
#   - important : key facts worth highlighting

ANOMALY_PROFILES = {

    "repetition": {
        "summary": (
            "This block shows excessive repetition of one or more events, "
            "far beyond what a healthy HDFS replication pipeline produces. "
            "The system is stuck in a retry loop rather than completing normally."
        ),
        "root_cause": (
            "The DataNode is repeatedly failing and retrying the same operation. "
            "This is typically caused by: unstable network connection between DataNodes "
            "causing packet loss and checksum failures, a DataNode disk that is slow or "
            "nearly full triggering repeated write retries, or a replication factor that "
            "cannot be satisfied because too few healthy DataNodes are available."
        ),
        "important": (
            "Retry storms like this consume significant DataNode CPU and network bandwidth. "
            "If left unresolved, the block may remain under-replicated and the NameNode "
            "will keep scheduling re-replication attempts, degrading cluster performance."
        ),
    },

    "missing_events": {
        "summary": (
            "This block is missing critical pipeline events that are present in every "
            "healthy HDFS block operation. The replication pipeline was cut short before "
            "it could complete — the block was never fully written or confirmed."
        ),
        "root_cause": (
            "The pipeline was aborted mid-execution. Common causes: a DataNode crashed "
            "or ran out of memory during the write, a network disconnection severed the "
            "replication pipeline between DataNodes, or the NameNode RPC timed out and "
            "the client gave up before all replicas were confirmed."
        ),
        "important": (
            "A block with missing pipeline events may be corrupt or only partially written. "
            "The NameNode may not have a confirmed replica for this block, which means "
            "the data it belongs to could be unreadable. Immediate inspection is advised."
        ),
    },

    "high_latency": {
        "summary": (
            "This block took significantly longer to complete than a normal HDFS block "
            "operation. While the pipeline eventually finished, the excessive time indicates "
            "a bottleneck or stall somewhere in the execution."
        ),
        "root_cause": (
            "Possible causes: network congestion between DataNodes causing pipeline stalls, "
            "disk I/O saturation on the writing DataNode slowing down block writes, "
            "JVM garbage collection stop-the-world pauses on a DataNode, or a DataNode "
            "under heavy CPU or memory pressure delaying packet processing."
        ),
        "important": (
            "High latency on individual blocks often signals a wider cluster health issue. "
            "If many blocks show similar latency, the affected DataNode may need to be "
            "decommissioned or the cluster may need additional capacity."
        ),
    },

    "duplicate_pattern": {
        "summary": (
            "This block contains events that repeat in a way inconsistent with the normal "
            "pipeline flow. Unlike a retry storm, the repetitions are moderate — the system "
            "was unstable but did not fully break down into a continuous retry loop."
        ),
        "root_cause": (
            "Possible causes: a client performed idempotent retries after a timeout without "
            "receiving acknowledgement, a race condition between two pipeline threads "
            "processing the same block simultaneously, or a partial network failure that "
            "caused some pipeline stages to be re-executed."
        ),
        "important": (
            "Duplicate patterns can leave the block in an ambiguous state — some replicas "
            "may have received the data twice while others received it once. This can cause "
            "inconsistency in block sizes across DataNodes."
        ),
    },

    "unknown": {
        "summary": (
            "This block shows an anomalous event sequence that does not match any of the "
            "known failure patterns. The sequence deviates from normal but the specific "
            "fault mechanism is not identifiable from the events alone."
        ),
        "root_cause": (
            "Root cause could not be determined automatically. The sequence may represent "
            "a rare or novel failure mode not captured in the current anomaly taxonomy."
        ),
        "important": (
            "Manual inspection of the full DataNode and NameNode logs for this BlockId "
            "is recommended to determine the actual cause."
        ),
    },
}